## Pretraining

### Load all allowed libraries

In [1]:
import numpy as np
from PIL import Image
import pandas as pd
import sklearn
import scipy
import seaborn as sns
import torch
import torchinfo
import torchvision
from tqdm import tqdm

import matplotlib.pyplot as plt

Could not save font_manager cache [Errno 28] No space left on device


### Define data transfromation function, and data augmentation

In [ ]:
from torchvision.transforms import v2

# Training data transforms and augmentation
training_tfms_and_agmt = v2.Compose([
    v2.ToImage(), # this turns it into a tensor

    v2.RandomRotation(degrees=10), # for realistic pet location variations 
    v2.RandomAffine(degrees=0, translate=(0.1, 0.1)), # for better position accuracy
    v2.RandomHorizontalFlip(p=0.5), # it is still the same cat if flip like this but double data

    v2.RandomResizedCrop((224, 224), scale=(0.8, 1.0), antialias=True), # 224 * 224 seems to be what everyone doing
    v2.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05), # this is for pet photograph varience

    v2.ToDtype(torch.float32, scale=True), # apply the min-max normalization and scale 0-255 to num between 0 and 1
    v2.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]), # apply the standardization for the z-scores things
])

valutation_tfms = v2.Compose([
    v2.ToImage(),
    v2.Resize(256, antialias=True),
    v2.CenterCrop(224),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

### Load dataset

In [3]:
from torchvision import datasets

trainval_data = datasets.OxfordIIITPet(
    root="/shared/storage/cs/studentscratch/cqh514",
    split='trainval',
    download=True,
)
img, label = trainval_data[0]

print(type(img))
print(img.mode)

<class 'PIL.Image.Image'>
RGB


### Define the split for the training dataset and the evaluation dataset

In [ ]:
num_total = len(trainval_data)
num_training = int(0.8 * num_total)
num_valuation = num_total - num_training

### Randomly split but make the split constant accross different runes

In [15]:
from torch.utils.data import random_split

generator = torch.Generator().manual_seed(67)
train_dataset, val_dataset = random_split(
    trainval_data, [num_training, num_valuation], generator=generator
)

### Create the data loaders

In [ ]:
from torch.utils.data import DataLoader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)

NameError: name 'DataLoader' is not defined